In [ ]:
# Practical 5: CBOW word embeddings from scratch (NumPy)
import re, numpy as np
np.random.seed(42)

# a. Data preparation
corpus = """The quick brown fox jumps over the lazy dog.
The dog barks at the fox and the fox runs away.
A quick brown dog jumps over the lazy fox.
Natural language processing helps computers understand human language.
Word embeddings capture the meaning of words in vectors.
The model learns word vectors by predicting a word from its context.
Computers learn language from large amounts of text.
The cat and the dog are friends and the cat likes the dog."""
tokens = re.sub(r"[^a-z\s]", " ", corpus.lower()).split()
vocab = sorted(set(tokens)); V = len(vocab); w2i = {w: i for i, w in enumerate(vocab)}
print(f"{len(tokens)} tokens, vocab size {V}")

In [ ]:
# b. Training data: (context word ids, target id), window = 2 each side
W = 2
data = [([w2i[tokens[j]] for j in range(max(0, i - W), min(len(tokens), i + W + 1)) if j != i], w2i[t])
        for i, t in enumerate(tokens)]
print(data[:2])

In [ ]:
# c. Train: average context embeddings -> hidden h -> softmax over vocab
N, EPOCHS, LR = 10, 500, 0.05
W1, W2 = np.random.randn(V, N) * .1, np.random.randn(N, V) * .1

def forward(ctx):
    h = W1[ctx].mean(0)                    # == average of one-hot vectors @ W1
    u = h @ W2; p = np.exp(u - u.max())
    return h, p / p.sum()

for ep in range(1, EPOCHS + 1):
    loss = 0
    for ctx, t in data:
        h, p = forward(ctx)
        loss -= np.log(p[t] + 1e-9)        # cross-entropy
        e = p.copy(); e[t] -= 1            # dL/du
        dh = W2 @ e                        # gradient reaching the hidden layer (before W2 changes)
        W2 -= LR * np.outer(h, e)
        np.add.at(W1, ctx, -LR * dh / len(ctx))   # only the context rows of W1 get gradient
    if ep == 1 or ep % 100 == 0: print(f"epoch {ep:3d}  loss {loss / len(data):.4f}")

In [ ]:
# d. Output: predict a word from context; nearest words by cosine similarity
def predict(words, k=3):
    _, p = forward([w2i[w] for w in words])
    return [(vocab[i], round(float(p[i]), 3)) for i in p.argsort()[::-1][:k]]

def similar(word, k=3):
    E = W1 / np.linalg.norm(W1, axis=1, keepdims=True); s = E @ E[w2i[word]]
    return [(vocab[i], round(float(s[i]), 3)) for i in s.argsort()[::-1][1:k + 1]]

print(predict(["the", "quick", "fox", "jumps"]))
print(predict(["the", "lazy", "the", "barks"]))
print(predict(["word", "capture", "meaning", "of"]))
print("fox vector:", W1[w2i["fox"]].round(2))
for w in ["fox", "language", "dog"]: print(w, similar(w))

In [ ]:
# (Manual's Keras version) Embedding + average = same CBOW, trained with Adam
from keras import Sequential, layers
from keras.optimizers import Adam
from keras.utils import pad_sequences
Xk = pad_sequences([[i + 1 for i in c] for c, _ in data], maxlen=2 * W)    # id 0 reserved for padding
yk = np.array([t for _, t in data])
cbow = Sequential([layers.Embedding(V + 1, N, mask_zero=True), layers.GlobalAveragePooling1D(),
                   layers.Dense(V, activation='softmax')])
cbow.compile(optimizer=Adam(0.01), loss='sparse_categorical_crossentropy')
cbow.fit(Xk, yk, epochs=200, batch_size=16, verbose=0)
print("Keras CBOW final loss:", round(cbow.evaluate(Xk, yk, verbose=0), 4))